# 04 Modeling: Price Prediction

Objective: build regression models to predict Airbnb listing price.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

from src.modeling import build_preprocessor
from src.evaluation import regression_metrics

DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PROCESSED / "modeling_airbnb_athens.csv")
df.head()

In [ ]:
target = "log_price"

numeric_features = [
    col for col in [
        "accommodates", "bathrooms", "bedrooms", "beds", "amenities_count",
        "minimum_nights", "availability_365", "availability_rate",
        "number_of_reviews", "review_scores_rating", "reviews_per_month",
        "host_experience_days"
    ] if col in df.columns
]

categorical_features = [
    col for col in [
        "neighbourhood_cleansed", "property_type", "room_type",
        "host_is_superhost", "instant_bookable", "is_entire_home"
    ] if col in df.columns
]

model_df = df[numeric_features + categorical_features + [target]].dropna(subset=[target])
X = model_df[numeric_features + categorical_features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
preprocessor = build_preprocessor(numeric_features, categorical_features)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001, max_iter=10000),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    metrics = regression_metrics(y_test, preds)
    metrics["Model"] = name
    results.append(metrics)

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

best_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", best_model)
])

best_pipe.fit(X_train, y_train)
joblib.dump(best_pipe, MODELS_DIR / "best_price_model.pkl")

best_model_name